# 🍽️ Meal Planning Assistant — Web Scraping Project

**Information Retrieval — Phase 1**

هذا المشروع يقوم بجمع 500+ وصفة من مواقع طعام عامة، تنظيفها، تحليلها، وبناء نظام توصيات قائم على القواعد.

## كيفية الاستخدام:
1. ارفع ملف `meal_planner_scraper.zip` (أيقونة الفولدر على الشمال → upload)
2. شغّل الخلايا واحدة واحدة بالترتيب (اضغط ▶️ على كل خلية)
3. خلية الـ scraping ستأخذ حوالي 17-20 دقيقة (احترام للموقع)

## 📦 الخطوة 1: فك ضغط الملف وتثبيت المكتبات

In [ ]:
!unzip -o meal_planner_scraper.zip
%cd scraper
!pip install -q -r requirements.txt
print('\n✅ التثبيت اكتمل')

## 🧪 الخطوة 2: اختبار سريع (قبل أي scraping حقيقي)
هذا يتأكد إن كل المكتبات شغالة والكود سليم. لازم يكتب في الآخر `✅ ALL CHECKS PASSED`

In [ ]:
!python test_pipeline.py

## 🌐 الخطوة 3: مسح الجلسة القديمة
بنمسح الـ 6 وصفات الوهمية من الاختبار عشان نبدأ بداتا نضيفة من الـ scraper الحقيقي

In [ ]:
import shutil, os
for folder in ['data/raw', 'data/recipes', 'data/plots']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)
for f in ['data/index.json', 'data/quality_report.json']:
    if os.path.exists(f):
        os.remove(f)
print('✅ تم مسح البيانات القديمة — جاهزين للـ scraping الحقيقي')

## 🕷️ الخطوة 4: جمع 500 وصفة (الـ Scraping الحقيقي)

**⏱️ هذه الخطوة ستأخذ حوالي 17-20 دقيقة**

نستخدم Food Network لأنه يعمل على Colab. كل طلب ننتظر ثانيتين بعده (احترام لسيرفرات الموقع — هذا جزء من الـ ethical scraping).

**حسبة الوقت:** 500 وصفة × 2 ثانية = 1000 ثانية ≈ 17 دقيقة

💡 **اتركها تشتغل في الخلفية** وارجع بعد 20 دقيقة.

In [ ]:
!python -m crawler.crawl --source foodnetwork --pages 30 --max-recipes 500 --delay 2

### تحقق سريع: كم وصفة جمعنا؟

In [ ]:
import glob
files = glob.glob('data/recipes/r_*.json')
print(f'✅ تم جمع {len(files)} وصفة')
if len(files) < 100:
    print('⚠️ العدد أقل من المتوقع. حاول تشغيل الخلية السابقة مرة تانية لإكمال العدد.')

## 🧹 الخطوة 5: تنظيف البيانات

**ثلاث خطوات في خلية واحدة:**
1. **Clean** — تحويل النصوص لأرقام، توحيد الوحدات، تحليل المكونات
2. **Enrich** — إضافة diet tags (vegetarian/vegan/gluten-free) من المكونات
3. **Quality** — معالجة المكرر، الناقص، والقيم الشاذة

In [ ]:
print('🧹 [1/3] التنظيف...')
!python -m pipeline.clean
print('\n🏷️  [2/3] إضافة الـ diet tags...')
!python -m pipeline.enrich
print('\n🔍 [3/3] فحص الجودة...')
!python -m pipeline.quality

## 📊 الخطوة 6: التحليل الاستكشافي (EDA)

نحسب الإحصائيات ونرسم 3 شارتس:
- أكثر المكونات استخداماً
- توزيع المطابخ (cuisines)
- توزيع أوقات الطبخ

In [ ]:
!python -m analysis.eda

## 🎨 الخطوة 7: عرض الشارتس داخل Colab

In [ ]:
from IPython.display import Image, display, Markdown

charts = [
    ('top_ingredients.png',      '📊 أكثر المكونات استخداماً'),
    ('cuisine_distribution.png', '🥘 توزيع المطابخ'),
    ('cooking_time_hist.png',    '⏱️ توزيع أوقات الطبخ'),
]

for filename, title in charts:
    display(Markdown(f'### {title}'))
    display(Image(filename=f'data/plots/{filename}'))

## 🔍 الخطوة 8: نظرة على وصفة واحدة (شكل الـ JSON)

In [ ]:
import json, glob

files = sorted(glob.glob('data/recipes/r_*.json'))
print(f'إجمالي الوصفات: {len(files)}\n')
print('مثال على وصفة واحدة (JSON):')
print('=' * 60)
sample = json.load(open(files[0]))
print(json.dumps(sample, indent=2, ensure_ascii=False)[:2000])
print('...')
print('=' * 60)

## 🤖 الخطوة 9: نظام التوصيات (Recommendation Engine)

غيّر القيم في `prefs` على حسب اللي عايزه واضغط ▶️.

**الخيارات:**
- `diet`: `"vegetarian"`, `"vegan"`, `"gluten-free"`, `"keto"`, أو `None` (بدون فلتر)
- `max_calories`: أقصى عدد سعرات حرارية للوجبة
- `max_time`: أقصى وقت بالدقايق
- `avoid`: قائمة بالمكونات اللي تتجنبها
- `pantry`: قائمة بالمكونات اللي عندك في البيت (تزيد الـ score)
- `top_n`: كم نتيجة تريد

In [ ]:
import sys
sys.path.insert(0, '.')
from app.cli import load_corpus, passes_hard_constraints, score_recipe

# 🎯 غيّر هذه القيم على حسب ما تريد
prefs = {
    'diet':         None,                              # جرّب "vegetarian" لاحقاً
    'max_calories': 800,
    'max_time':     60,
    'avoid':        [],                                 # مثال: ['mushroom', 'cilantro']
    'pantry':       ['butter', 'sugar', 'flour', 'egg'],
    'top_n':        10,
}

# تحميل الكوربس
corpus = load_corpus()

# الفلترة (FILTER + EXCLUDE)
candidates = []
for r in corpus:
    ok, trace = passes_hard_constraints(r, prefs)
    if ok:
        candidates.append((r, trace))

# الترتيب (SCORE + RANK)
ranked = sorted(
    ((r, t, score_recipe(r, prefs)) for r, t in candidates),
    key=lambda x: x[2], reverse=True,
)

# عرض النتائج
print('=' * 70)
print(f'النتائج: {len(corpus)} وصفة → {len(candidates)} مطابقة → أفضل {prefs["top_n"]}')
print('=' * 70)

for i, (r, trace, score) in enumerate(ranked[:prefs['top_n']], 1):
    title = r.get('title', '(بدون عنوان)')
    cal = r.get('calories_per_serving') or '?'
    t = r.get('total_time_min') or '?'
    diet = ', '.join(r.get('diet_tags') or []) or '—'
    print(f'\n  {i}. {title}')
    print(f'     🔥 {cal} cal  |  ⏱️ {t} min  |  🏷️ {diet}')
    print(f'     ⭐ score: {score:.2f}')
    if trace:
        print(f'     ✓ matched: {", ".join(trace.keys())}')

if not ranked:
    print('\n❌ لا توجد نتائج — جرّب تخفيف الفلاتر')

## 💾 الخطوة 10 (اختياري): تحميل الداتا على جهازك

Colab سيمسح كل الفايلات لما تقفل الجلسة. لو تريد تحتفظ بالداتا، شغّل هذه الخلية لتحميل zip على جهازك.

In [ ]:
import shutil
from google.colab import files

# اعمل zip للداتا والشارتس
shutil.make_archive('my_recipes_dataset', 'zip', 'data')
files.download('my_recipes_dataset.zip')

## 💡 نصايح وإجابات أسئلة شائعة

### ❓ الـ scraping بطيء جداً!
هذا **مقصود وصحيح**. الـ 2 ثانية بين كل طلب جزء أساسي من الـ ethical scraping (ذكرناه في العرض التقديمي slide 8). لو شغّلته بدون انتظار، ممكن:
- يسبب ضغط على سيرفرات الموقع
- الموقع يبلوكنا (والمشاريع التانية اللي تيجي بعدنا)
- نخسر درجات في معيار "Robots.txt compliance"

### ❓ بعض المواقع تظهر `403 Forbidden` أو `404 Not Found`
هذا طبيعي على Colab. سيرفرات Colab معروفة كـ data centers، فبعض المواقع تبلوكها تلقائياً. الحل: استخدم `--source foodnetwork` فقط على Colab، أو شغّل المشروع على جهازك المحلي.

### ❓ كيف أزود مصادر للداتا؟
افتح `crawler/crawl.py` وأضف مصدر جديد في dictionary `SOURCES`. كل مصدر يحتاج: name, host, listing_url function, recipe_link_filter function.

### ❓ كيف أعيد التنظيف لو غيّرت الكود؟
`pipeline/clean.py` آمن للتشغيل عدة مرات (idempotent). فقط شغّل الخلية 5 مرة تانية.

### ❓ أين أجد الـ raw HTML؟
في `data/raw/` — كل صفحة محفوظة هناك بحيث تقدر تعيد التحليل بدون scraping تاني (يوفر وقت كبير لما تطور الكود).

---

🎓 **مشروع IR Phase 1 — Meal Planning Assistant**